# Fetching Census Data

This notebook shows how `mxcensus` fetches data on demand from the parquet mirror hosted on GitHub Releases.

Pooch downloads each file the first time it is requested and caches it locally — subsequent calls are instant.

The mirror covers all of Mexico's 32 states across three data families:

| Family | Files | Loader | Description |
|--------|-------|--------|-------------|
| **Census tabular** | `iter`, `resargebub`, `personas`, `viviendas` | `load_census`, `load_extended_personas`, `load_extended_viviendas` | Locality / AGEB-block aggregates and extended-questionnaire microdata |
| **Marco Geoestadístico** | `mg_*` (15 layers) | `load_mg_census` | INEGI geometries merged with census attributes |
| **DENUE** | `denue_{YYYYMM}_*` (25 releases, 2010–2026) | `load_denue` | Economic-unit point microdata, longitudinally harmonizable |

## 1. Where does the data go?

Files are cached in a platform-specific directory managed by `platformdirs`.  
Override it with the `MXCENSUS_CACHE_DIR` environment variable.

In [ ]:
from mxcensus.data import get_pooch_cache_dir, POOCH

print(f"Cache directory : {get_pooch_cache_dir()}")
print(f"Mirror base URL : {POOCH.base_url}")
print(f"Registered files: {len(POOCH.registry)}")

## 2. Load aggregate census data

`load_census(state=N)` downloads the raw ITER and RESARGEBUB parquets on the first call, then runs the full processing pipeline (imputation, sanity checks, derived columns) and returns four DataFrames.

In [ ]:
from mxcensus import load_census

# State 9 = Ciudad de México
df_state, df_mun, df_loc, df_ageb = load_census(state=9)

print("State-level shape :", df_state.shape)
print("Municipality shape:", df_mun.shape)
print("Locality shape    :", df_loc.shape)
print("AGEB shape        :", df_ageb.shape)

## 3. Load extended questionnaire microdata

In [ ]:
from mxcensus import load_extended_personas, load_extended_viviendas

df_personas  = load_extended_personas(state=9)
df_viviendas = load_extended_viviendas(state=9)

print("Personas shape :", df_personas.shape)
print("Viviendas shape:", df_viviendas.shape)

## 4. Load Marco Geoestadístico geometries

`load_mg_census(state=N)` fetches the four MGN geoparquet layers it needs (`mg_a`, `mg_l`, `mg_lpr`, `mg_ar`), loads the census via `load_census`, and runs the geometry-merge pipeline. It returns two `GeoDataFrame`s: the urban / aggregated-rural AGEBs (`mg_aur`) and the locality/AGEB geometries with census attributes (`mg_loc_ageb`).

In [ ]:
from mxcensus import load_mg_census

mg_aur, mg_loc_ageb = load_mg_census(state=9)

print("Urban/aggregated-rural AGEBs:", mg_aur.shape, "| CRS:", mg_aur.crs)
print("Locality/AGEB geometries    :", mg_loc_ageb.shape)
mg_aur.head(3)

## 5. Load DENUE economic units

DENUE (Directorio Estadístico Nacional de Unidades Económicas) is published periodically; `mxcensus` mirrors **25 releases (2010–2026)** as point geoparquet (EPSG:4326).

`load_denue(state=N)` fetches the **latest** release by default; pass `release="YYYYMM"` for a specific edition.

DENUE's schema drifts across releases (columns renamed, recoded, reshuffled into 11 distinct "schema groups"). The `harmonize` flag controls which schema you get back:

- **`harmonize=True`** (default) — maps any release onto the latest 42-field schema (plus the `geometry` column), canonicalizing `per_ocu`, `tipoUniEco`, and `fecha_alta` so releases are **longitudinally comparable**.
- **`harmonize=False`** — returns the release's **raw** columns, exactly as INEGI published them.

In both cases the frame is validated against a tight Pandera schema; value-level violations (e.g. INEGI's known corrupt postal codes) emit a `warnings.warn` summary rather than raising.

In [ ]:
from mxcensus import load_denue

# Harmonized (default): the canonical 42-field schema + a geometry column
denue_h = load_denue(state=9)

print("Harmonized shape  :", denue_h.shape)
print("CRS               :", denue_h.crs)
print("Columns           :", len(denue_h.columns), "(42 harmonized fields + geometry)")
print(list(denue_h.columns))
denue_h[["nom_estab", "codigo_act", "per_ocu", "tipoUniEco", "fecha_alta"]].head(3)

In [ ]:
# Raw: the release's own columns, untouched
denue_raw = load_denue(state=9, harmonize=False)

print("Raw shape   :", denue_raw.shape)
print("Raw columns :", list(denue_raw.columns))

# What harmonization changed for this (already-latest) release:
print("\nColumns only in raw       :", sorted(set(denue_raw.columns) - set(denue_h.columns)))
print("Columns only in harmonized:", sorted(set(denue_h.columns) - set(denue_raw.columns)))

### Harmonization across releases

The payoff is comparing editions years apart. An **old** release (e.g. 2012) has different column names, a coded `per_ocu`, and a numeric `Tipo de establecimiento` — yet harmonized it lands on the *same* fields and the *same* canonical category labels as the latest edition.

In [ ]:
# Same state, an old release, raw vs harmonized
denue_2012_raw = load_denue(state=9, release="201200", harmonize=False)
denue_2012_h   = load_denue(state=9, release="201200")  # harmonized

print("2012 raw columns       :", list(denue_2012_raw.columns))
print("\n2012 harmonized == latest harmonized schema:",
      list(denue_2012_h.columns) == list(denue_h.columns))

# Canonical categories line up across eras
print("\nper_ocu    (2012):", sorted(denue_2012_h["per_ocu"].dropna().unique()))
print("per_ocu    (2026):", sorted(denue_h["per_ocu"].dropna().unique()))
print("tipoUniEco (2012):", sorted(denue_2012_h["tipoUniEco"].dropna().unique()))
print("fecha_alta (2012):", sorted(denue_2012_h["fecha_alta"].dropna().unique())[:3], "(YYYY-MM)")

## 6. Pre-fetch without loading

Use `POOCH.fetch()` directly to download a file to the cache without reading it into memory.  
The CLI equivalent is `mxcensus fetch 9` (add `--dataset denue --release 202605` for DENUE).

In [ ]:
from mxcensus.data import POOCH
from mxcensus.data._catalog import STATE_CODE_FMT

code = STATE_CODE_FMT(9)
for ds in ("iter", "resargebub", "personas", "viviendas"):
    path = POOCH.fetch(f"{ds}_{code}.parquet", progressbar=True)
    print(f"{ds}_{code}.parquet → {path}")

# DENUE files are release-qualified: denue_{YYYYMM}_{NN}.parquet
print(POOCH.fetch(f"denue_202605_{code}.parquet", progressbar=True))

## 7. Load from an explicit path

All load functions accept an explicit file path for users who have their own raw parquets or CSVs.

In [ ]:
from pathlib import Path
from mxcensus import load_census, load_denue

# Census: both CSV and parquet are accepted
# df_state, df_mun, df_loc, df_ageb = load_census(
#     iter_path=Path("path/to/iter_09.parquet"),
#     resargebub_path=Path("path/to/resargebub_09.parquet"),
# )

# DENUE: point a survey_path at a local geoparquet
# denue = load_denue(survey_path=Path("data/parquet/denue_202605_09.parquet"))

## 8. State codes reference

INEGI uses two-digit codes (ENTIDAD) to identify states.

In [ ]:
import pandas as pd
from mxcensus.data._catalog import STATE_ABBR, STATE_CODE_FMT

STATE_NAMES = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur",
    4: "Campeche", 5: "Coahuila", 6: "Colima", 7: "Chiapas",
    8: "Chihuahua", 9: "Ciudad de México", 10: "Durango",
    11: "Guanajuato", 12: "Guerrero", 13: "Hidalgo", 14: "Jalisco",
    15: "Estado de México", 16: "Michoacán", 17: "Morelos", 18: "Nayarit",
    19: "Nuevo León", 20: "Oaxaca", 21: "Puebla", 22: "Querétaro",
    23: "Quintana Roo", 24: "San Luis Potosí", 25: "Sinaloa", 26: "Sonora",
    27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz",
    31: "Yucatán", 32: "Zacatecas",
}

df = pd.DataFrame([
    {"code": STATE_CODE_FMT(c), "abbreviation": abbr, "name": STATE_NAMES[c]}
    for c, abbr in STATE_ABBR.items()
])
df